In [4]:
#imports
import pandas as pd
import numpy as np
import os
from datetime import datetime
import glob


In [5]:
#folders path
folder_path = r"D:\Graduation Project\export\apple_health_export\electrocardiograms"
output_file = r"D:\Graduation Project\ecg_readings_with_timestamps.csv"

In [6]:
#initialization of empty lists to store data
all_data = []


In [7]:
for csv_file in glob.glob(os.path.join(folder_path, "*.csv")):
    try:
        with open(csv_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        lines = content.strip().split('\n')
        
        # Extract recording datetime
        recording_datetime = None
        for line in lines[:20]: #check only first 20 lines for metadata 3lshan na5od menha el date w el time
            if 'Recorded Date' in line:
                date_str = line.split(',', 1)[1].strip()
                try:
                    date_part = date_str.split('+')[0].strip()
                    recording_datetime = datetime.strptime(date_part, '%Y-%m-%d %H:%M:%S')
                except:
                    try:
                        recording_datetime = datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S')
                    except:
                        pass
                break
        
        # Extract sample rate 3lshan a3rf a3ml timestamps 3lshan el ecg readings
        sample_rate = 512  
        for line in lines[:20]:
            if 'Sample Rate' in line:
                try:
                    rate_str = line.split(',', 1)[1].strip()
                    sample_rate = float(rate_str.split()[0])
                except:
                    pass
                break
        
        # Find ECG data start
        ecg_start = 0
        for i, line in enumerate(lines):
            if 'Unit,µV' in line or 'Unit,mV' in line:
                ecg_start = i + 1
                break
        
        # Extract ECG readings
        ecg_readings = []
        for line in lines[ecg_start:]:
            if line.strip() and ',' not in line:
                try:
                    ecg_readings.append(float(line.strip()))
                except:
                    continue
        
        if not ecg_readings or recording_datetime is None:
            print(f"Skipping {os.path.basename(csv_file)} - no readings or invalid date")
            continue
        
        # Create timestamps
        timestamps = [recording_datetime + pd.Timedelta(seconds=i/sample_rate) for i in range(len(ecg_readings))] # Generate timestamps based on sample rate 3lshan a3ml el time index
        
        # Create DataFrame
        file_df = pd.DataFrame({
            'timestamp': timestamps,
            'ecg': ecg_readings
        })
        
        all_data.append(file_df)
        print(f"Processed {os.path.basename(csv_file)}: {len(ecg_readings)} readings starting at {recording_datetime}")
        
    except Exception as e:
        print(f"Error with {csv_file}: {str(e)}")
        continue


Processed ecg_2023-09-22.csv: 15360 readings starting at 2023-09-22 14:15:10
Processed ecg_2023-10-11.csv: 15360 readings starting at 2023-10-11 13:04:24
Processed ecg_2023-10-11_1.csv: 15360 readings starting at 2023-10-11 13:08:51
Processed ecg_2023-10-20.csv: 15360 readings starting at 2023-10-20 01:48:46
Processed ecg_2023-12-04.csv: 15360 readings starting at 2023-12-04 13:15:29
Processed ecg_2024-01-01.csv: 15360 readings starting at 2024-01-01 23:26:35
Processed ecg_2024-09-13.csv: 15360 readings starting at 2024-09-13 03:31:11
Processed ecg_2024-12-29.csv: 15360 readings starting at 2024-12-29 14:14:37
Processed ecg_2025-04-19.csv: 15360 readings starting at 2025-04-19 21:50:21
Processed ecg_2025-04-19_1.csv: 15360 readings starting at 2025-04-19 21:52:47
Processed ecg_2025-04-19_2.csv: 15360 readings starting at 2025-04-19 21:53:56
Processed ecg_2025-05-01.csv: 15360 readings starting at 2025-05-01 19:36:51
Processed ecg_2025-08-08.csv: 15360 readings starting at 2025-08-08 16

In [8]:
if all_data:
    dataset = pd.concat(all_data, ignore_index=True)
    dataset = dataset.sort_values('timestamp')
else:
    dataset = pd.DataFrame()


In [9]:
if not dataset.empty:
    dataset.to_csv(output_file, index=False)
    print(f"\n SUCCESS! Saved {len(dataset):,} ECG readings to {output_file}")
else:
    print(" No data was processed. Check the folder path and file formats.")



 SUCCESS! Saved 215,040 ECG readings to D:\Graduation Project\ecg_readings_with_timestamps.csv
